In [1]:
import pandas as pd
import json
import plotly.express as px
import numpy as np
import seaborn as sns

In [2]:
with open('./user_course_info_cleaned.json', 'r') as file:
    user_course_info = json.load(file)

df_user_course_info = pd.DataFrame(user_course_info)

In [3]:
with open('./teacher_stats_cleaned.json', 'r') as file:
    teacher_stats = json.load(file)

df_teacher_stats = pd.DataFrame(teacher_stats)

In [4]:
with open('./pro_course_cleaned.json', 'r') as file:
    pro_course = json.load(file)

df_pro_course = pd.DataFrame(pro_course)

In [5]:
with open('./price_list_cleaned.json', 'r') as file:
    price_list = json.load(file)

df_price_list = pd.DataFrame(price_list)

In [6]:
with open('./also_speaks_cleaned.json', 'r') as file:
    also_speaks = json.load(file)

df_also_speaks = pd.DataFrame(also_speaks)

In [7]:
with open('./also_speaks_reference.json', 'r') as file:
    also_speaks_reference = json.load(file)

df_also_speaks_reference = pd.DataFrame(also_speaks_reference)

Number of Records in Each Table

In [8]:
dataframe_list = [df_user_course_info, df_teacher_stats, df_pro_course, df_price_list, df_also_speaks, df_also_speaks_reference]
def get_all_df(dataframe_list):
    for df in dataframe_list:
        df.info()
        print()

In [9]:
dataframe_items = ['User Course Info', 'Teacher Stats', 'Pro Course', 'Price List', 'Also Speaks', 'Also Speaks Reference']
retrieved_languages = ['english', 'chinese', 'french', 'spanish', 'japanese', 'italian', 'german', 'korean']

In [10]:
len(dataframe_list[0])

10470

In [11]:
records = []
records_dict = {}
for item in dataframe_list:
    records.append(item.shape)
print(records)

for i in range(len(dataframe_list)):
    records_dict[dataframe_items[i]] = records[i]
records_dict

df_records_dict = pd.DataFrame(records_dict)
df_records_dict = df_records_dict.rename(index={0: 'Rows', 1: 'Columns'})
df_records_dict

[(10470, 17), (10470, 4), (39035, 8), (215827, 6), (29185, 2), (210, 1)]


,User Course Info,Teacher Stats,Pro Course,Price List,Also Speaks,Also Speaks Reference
Rows,10470,10470,39035,215827,29185,210
Columns,17,4,8,6,2,1


How many teachers of each language are there?\
During the data collection, the following languages were collected:\
English, Chinese, French, Spanish, Japanese, Italian, German, Korean\
However, since the API call is by teacher, there may be other languages taught in the dataset

In [12]:
df_teacher_language = df_pro_course.groupby(['teacher_id', 'language']).count().reset_index()
df_teacher_language_count = df_teacher_language.groupby('language')[['teacher_id']].count()
print(f'There are a total of {len(df_teacher_language_count)} languages taught')
pro_course_total = df_teacher_language_count.sum().iloc[0]
print(f'There are a total of {pro_course_total} pro courses taught')


There are a total of 147 languages taught
There are a total of 13701 pro courses taught


In [13]:
df_top_20 = df_teacher_language_count.sort_values('teacher_id', ascending=False).head(20).reset_index()
df_top_20

,language,teacher_id
0,english,4180
1,spanish,2241
2,french,1170
3,japanese,1115
4,chinese,998
5,italian,818
6,german,683
7,korean,538
8,russian,214
9,portuguese,173


Although only teachers of 8 languages were retrieved from the platform, there are another 139 languages taught among these teachers.



In [14]:
fig = px.bar(df_top_20, x='language', y='teacher_id', title='Number of Teachers by Language Taught (Top 20)', labels={'teacher_id': 'Number of Teachers', 'language': 'Language'})
fig.show()

What is the mean hourly price for each of languages retrieved?

In [15]:
get_all_df(dataframe_list)

<class 'pandas.DataFrame'>
RangeIndex: 10470 entries, 0 to 10469
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   user_id              10470 non-null  int64
 1   nickname             10470 non-null  str  
 2   is_tutor             10470 non-null  int64
 3   is_pro               10470 non-null  int64
 4   origin_country_id    10470 non-null  str  
 5   living_country_id    10470 non-null  str  
 6   origin_city_id       10470 non-null  str  
 7   origin_city_name     10470 non-null  str  
 8   living_city_id       10470 non-null  str  
 9   living_city_name     10470 non-null  str  
 10  timezone             10470 non-null  str  
 11  trial_length         10470 non-null  int64
 12  has_trial            10470 non-null  int64
 13  trial_price          10470 non-null  int64
 14  min_price            10470 non-null  int64
 15  trial_session_count  10470 non-null  int64
 16  has_beginner_course  10470 non-nu

In [16]:
average_price = {}
median_price = {}

for language in retrieved_languages:
    median = df_pro_course[df_pro_course['language'] == language]['session_price'].median()
    mean = df_pro_course[df_pro_course['language'] == language]['session_price'].mean()
    average_price[language] = round(mean)/100
    median_price[language] = round(median)/100

df_average_price = pd.DataFrame.from_dict(average_price, orient="index", columns=["Average Price"])
df_average_price
fig = px.bar(df_average_price, y='Average Price')
fig.show()


In [17]:
df_median_price = pd.DataFrame.from_dict(median_price, orient="index", columns=["Median Price"])
df_median_price

,Median Price
english,22.0
chinese,22.0
french,25.0
spanish,18.0
japanese,24.0
italian,25.0
german,32.0
korean,21.0


In [18]:

fig = px.bar(df_median_price, y='Median Price')
fig.show()

It is surprising that both the Median and Mean price of German is the highest among all language teachers.\
Further investigation could be conducted to find if there is more to it.

In [19]:
df_pro_course[df_pro_course['id'] == 290784]

,id,teacher_id,language,title,session_price,student_count,session_count,has_package
39030,290784,31481128,french,🗣️ Natural French Conversation: Speak easily !,700,0,0,1


In [20]:
df_price_list[df_price_list['session_length'] == 4]

,package_price,session_price,course_id,package_length,session_length,course_price_id
2,20900,2300,946,10,4,823
5,30500,3200,4042,10,4,3934
7,30900,3400,5620,10,4,5525
8,20900,2300,5779,10,4,5686
13,15000,1590,8775,10,4,8797
...,...,...,...,...,...,...
215805,55000,4000,290796,15,4,1084922
215809,70000,4000,290796,20,4,1084926
215810,38000,4000,290796,10,4,1084927
215825,59999,7498,87801,10,4,1085029


How many teachers teach multiple languages?

In [21]:
df_multiple_languages = df_pro_course.groupby(['teacher_id','language']).count().reset_index()
df_multiple_languages = df_multiple_languages[['teacher_id', 'language']].groupby('teacher_id').count()
df_multiple_languages = df_multiple_languages.reset_index()
count = len(df_multiple_languages[df_multiple_languages['language'] > 1])
print(f'{round(count/len(df_multiple_languages) * 100, 2)}% of teachers teach multiple languages')

22.37% of teachers teach multiple languages


Among the teachers that teach multiple languages, what are the two languages that a single teacher is most likely to teach?

In [22]:
df_top_8_languages = df_teacher_language_count.sort_values('teacher_id', ascending=False).head(8).reset_index()
row_list = df_top_8_languages['language'].values.tolist()
row_list

['english',
 'spanish',
 'french',
 'japanese',
 'chinese',
 'italian',
 'german',
 'korean']

In [23]:
df_multiple_languages[df_multiple_languages['language'] > 1]

,teacher_id,language
5,222305,5
6,249152,2
13,399468,2
15,436016,2
17,468130,2
...,...,...
10391,31822868,2
10393,31823132,2
10396,31826538,2
10409,31913530,2


In [24]:
df_multiple_languages_original = df_pro_course.groupby(['teacher_id','language']).count().reset_index()
df_multiple_languages_original = df_multiple_languages_original[['teacher_id', 'language']]
df_multiple_languages_original

,teacher_id,language
0,55502,chinese
1,113638,chinese
2,114708,chinese
3,132815,korean
4,148092,spanish
...,...,...
13696,32167939,haitiancreole
13697,32168520,english
13698,32169200,english
13699,32169731,english


In [25]:
df_multiple_dummies = pd.get_dummies(df_multiple_languages_original, columns=['language'])
df_multiple_dummies = df_multiple_dummies.groupby('teacher_id').max().reset_index()
df_multiple_dummies_8 = df_multiple_dummies[['teacher_id', 'language_english', 'language_chinese', 'language_french', 'language_spanish', 'language_german', 'language_japanese', 'language_korean', 'language_italian']]
df_english = df_multiple_dummies_8[df_multiple_dummies_8['language_english'] == True]
df_english_sum = df_english[['language_chinese', 'language_french', 'language_spanish', 'language_german', 'language_japanese', 'language_korean', 'language_italian']].sum().reset_index()
df_english_sum = df_english_sum.rename(columns={'index': 'language', 0: 'count'})
df_english_sum

,language,count
0,language_chinese,118
1,language_french,238
2,language_spanish,368
3,language_german,118
4,language_japanese,48
5,language_korean,36
6,language_italian,144


In [26]:
fig = px.bar(df_english_sum, x='language', y='count', title='Count of English Teachers that also teach other popular languages', labels={'count': 'Number of Teachers', 'language': 'Language'})
fig.show()

The conclusion is not very surprising. The highest combination must be English and another language.\
The maximum number of a language combination is limited by the number of the total count of the language with the lower count.\
Hence it must be English and another language. Since the second most number of language teachers are Spanish teachers, the highest combination is English-Spanish.

What cities do teachers of the various languages live in?

In [27]:
df_english_teachers = df_pro_course[df_pro_course['language'] == 'english'].groupby(['teacher_id', 'language']).count()
df_english_teachers = df_english_teachers.reset_index()
df_english_teachers = df_english_teachers[['teacher_id']]
df_english_teachers

,teacher_id
0,222305
1,346460
2,399468
3,426279
4,436016
...,...
4175,32165182
4176,32166542
4177,32168520
4178,32169200


In [28]:
df_merged = pd.merge(df_english_teachers, df_user_course_info, left_on='teacher_id', right_on='user_id', how='left')
df_city = df_merged[['teacher_id', 'living_city_id']].groupby('living_city_id').count()
df_city = df_city.reset_index()
df_city.sort_values('teacher_id', ascending=False).head(50)

,living_city_id,teacher_id
982,ZZ00000,251
318,GB00001,151
946,ZA00001,83
317,GB00000,70
948,ZA00011,69
789,TR00000,67
820,US00000,58
945,ZA00000,49
891,US00831,45
637,PH00000,44


In [29]:
df_merged = pd.merge(df_english_teachers, df_user_course_info, left_on='teacher_id', right_on='user_id', how='left')
df_city = df_merged[['teacher_id', 'living_city_name']].groupby('living_city_name').count()
df_city = df_city.reset_index()
df_city.sort_values('teacher_id', ascending=False).head(50)

,living_city_name,teacher_id
587,Other,874
441,London,152
358,Johannesburg,83
144,Cape Town,69
547,New York,45
809,Toronto,40
124,Buenos Aires,33
496,Mexico City,33
65,Barcelona,32
601,Paris,32


Is there a big difference between the average price between tutors and pros?

Check if there are teachers who are both tutors and professional teachers.\
italki's website says that they are not supposed to be both.\
https://support.italki.com/hc/en-us/articles/900002978766-Difference-between-Professional-Teacher-and-Community-Tutor

Doing a sample check and matching with the italki website, if both is_tutor and is_pro are marked as 1, the is_pro takes priority and the user is a professional teacher

In [30]:
df_tutors = df_user_course_info[(df_user_course_info['is_tutor'] == 1) & (df_user_course_info['is_pro']  == 0)]
df_tutors

,user_id,nickname,is_tutor,is_pro,origin_country_id,living_country_id,origin_city_id,origin_city_name,living_city_id,living_city_name,timezone,trial_length,has_trial,trial_price,min_price,trial_session_count,has_beginner_course
0,55502,Micky Mick,1,0,CN,CN,CN00226,Fushun,CN00006,Beijing,Asia/Shanghai,2,0,1000,2000,97,1
3,132815,Julie 줄리,1,0,KR,KR,KR00001,Seoul,KR00001,Seoul,Asia/Seoul,2,0,750,1500,0,1
6,249152,CeciliaStockholm,1,0,AR,SE,ZZ00000,Other,SE00061,Boras,Europe/Stockholm,2,0,800,1800,70,0
11,355886,Laya,1,0,VE,VE,ZZ00000,Other,ZZ00000,Other,America/Caracas,2,0,500,750,142,1
12,368484,Angeles,1,0,ES,ES,ES00181,Burgos,ES00181,Burgos,Europe/Madrid,2,0,1000,2100,91,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10465,32167939,Andrele,1,0,CA,CA,CA00000,Other,CA00000,Other,America/Toronto,2,0,1000,2000,3,1
10466,32168520,Daphne👩🏫🌸💗,1,0,PH,PH,PH00141,Baguio,PH00000,Other,Asia/Manila,3,0,500,700,0,0
10467,32169200,Davide Castellani,1,0,IT,IT,IT00000,Other,IT00000,Other,Europe/Rome,4,0,1400,1400,0,1
10468,32169731,FelixRansford,1,0,GB,ES,GB00281,Exeter,ES00131,Jerez de la Frontera,Europe/Madrid,2,0,600,1400,0,0


Is there a difference in average or median price between tutors and pro's prices per language

In [31]:
df_pro_course_english = df_pro_course[df_pro_course['language'] == 'english']
df_pro_course_english

,id,teacher_id,language,title,session_price,student_count,session_count,has_package
0,375,436016,english,English courses新概念英语（1）,1400,15,106,0
4,4042,463477,english,One-on-One tutoring: English,3200,95,1504,1
6,5779,502096,english,One-on-One tutoring: English,2300,84,876,1
17,10048,596847,english,English language instruction,2600,147,2881,1
21,10779,483024,english,English Pronunciation & Intonation,6500,49,90,1
...,...,...,...,...,...,...,...,...
39014,290713,18491956,english,IELTS Preparation Course,800,0,0,1
39015,290714,18491956,english,"Medical, Health & Fitness English",800,0,0,1
39017,290721,5846271,english,"Beginner English (for FR, ES, PT, ZH speakers)",2200,0,0,1
39019,290736,19100180,english,English Language Topics Lesson,2300,0,0,1


In [32]:
df_tutors_users = df_tutors[['user_id']]
df_tutors_users

,user_id
0,55502
3,132815
6,249152
11,355886
12,368484
...,...
10465,32167939
10466,32168520
10467,32169200
10468,32169731


In [33]:
df_merged = pd.merge(df_tutors, df_pro_course_english, left_on="user_id", right_on="teacher_id", how="inner")
df_merged

,user_id,nickname,is_tutor,is_pro,origin_country_id,living_country_id,origin_city_id,origin_city_name,living_city_id,living_city_name,...,trial_session_count,has_beginner_course,id,teacher_id,language,title,session_price,student_count,session_count,has_package
0,484682,Rick,1,0,CA,US,CA00001,Toronto,US00136,Boise City,...,0,0,164410,484682,english,Conversation Practice,1600,13,238,1
1,495195,Maria Exam Work etc,1,0,RU,RU,RU00001,Moscow,RU00001,Moscow,...,440,0,115395,495195,english,Effective method to study English for tests,1400,41,250,1
2,495195,Maria Exam Work etc,1,0,RU,RU,RU00001,Moscow,RU00001,Moscow,...,440,0,131082,495195,english,English classes,1400,20,108,1
3,495195,Maria Exam Work etc,1,0,RU,RU,RU00001,Moscow,RU00001,Moscow,...,440,0,224953,495195,english,Conversational lesson with the focus on learning,1400,17,122,1
4,544865,Nalini,1,0,IN,IT,IN00231,Delhi,IT00206,Novara,...,0,0,159180,544865,english,English Conversation Practice,1800,80,934,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4616,32168520,Daphne👩🏫🌸💗,1,0,PH,PH,PH00141,Baguio,PH00000,Other,...,0,0,290359,32168520,english,Let’s Chat,700,1,1,1
4617,32168520,Daphne👩🏫🌸💗,1,0,PH,PH,PH00141,Baguio,PH00000,Other,...,0,0,290362,32168520,english,Reading English,700,0,0,1
4618,32168520,Daphne👩🏫🌸💗,1,0,PH,PH,PH00141,Baguio,PH00000,Other,...,0,0,290365,32168520,english,English for Grammar,800,0,0,1
4619,32169200,Davide Castellani,1,0,IT,IT,IT00000,Other,IT00000,Other,...,0,1,290327,32169200,english,English Conversation Practice for Everyday Sit...,1400,0,0,0


In [34]:
median = df_merged['session_price'].mean()
median

np.float64(1845.064488206016)

These are the average and median prices of tutors for the 8 languages examined

In [35]:
average_price = {}
median_price = {}

for language in retrieved_languages:
    df_pro_course_language = df_pro_course[df_pro_course['language'] == language]
    df_merged = pd.merge(df_tutors, df_pro_course_language, left_on="user_id", right_on="teacher_id", how="inner")
    median = df_merged['session_price'].median()
    mean = df_merged['session_price'].mean()
    average_price[language] = round(mean)/100
    median_price[language] = round(median)/100

df_average_price = pd.DataFrame.from_dict(average_price, orient="index", columns=["Average Price"])
df_average_price
fig = px.bar(df_average_price, y='Average Price')
fig.show()


In [36]:
df_median_price = pd.DataFrame.from_dict(median_price, orient="index", columns=["Median Price"])
df_median_price
fig = px.bar(df_median_price, y='Median Price')
fig.show()


What is the average and median price of professional teachers?

In [37]:
df_pros_1 = df_user_course_info[(df_user_course_info['is_tutor'] == 1) & (df_user_course_info['is_pro']  == 1)]
df_pros_1

,user_id,nickname,is_tutor,is_pro,origin_country_id,living_country_id,origin_city_id,origin_city_name,living_city_id,living_city_name,timezone,trial_length,has_trial,trial_price,min_price,trial_session_count,has_beginner_course
1,113638,Xin,1,1,CN,CA,,,CA00006,Montreal,America/Toronto,2,0,999,2500,227,0
5,222305,Adriano teacher,1,1,BR,BR,ZZ00000,Other,ZZ00000,Other,America/Sao_Paulo,2,0,799,1499,1378,0
7,259649,Ally,1,1,CN,CN,ZZ00000,Other,ZZ00000,Other,Asia/Shanghai,2,0,690,1190,0,0
8,291538,RYOTA,1,1,JP,JP,JP00011,Osaka,JP00001,Tokyo,Asia/Phnom_Penh,2,0,650,2480,0,0
9,316582,CLAU📚S☧anish📖💛💙❤,1,1,VE,VE,VE00016,Barquisimeto,VE00016,Barquisimeto,America/Caracas,2,0,500,1000,148,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10073,30619874,MISO,1,1,KR,JP,KR00116,Jeju,JP00006,Yokohama,Asia/Tokyo,2,0,500,1600,0,0
10170,30955232,Aleksandra,1,1,GB,CN,GB00031,Manchester,CN00001,Shanghai,Asia/Shanghai,2,0,725,1500,0,0
10251,31286123,Yuko,1,1,JP,JP,JP00011,Osaka,JP00011,Osaka,Asia/Tokyo,3,0,900,1900,10,1
10299,31472940,margot,1,1,FR,FR,FR00000,Other,FR00000,Other,Europe/Paris,2,0,500,4000,0,1


In [38]:
df_pros_2 = df_user_course_info[(df_user_course_info['is_tutor'] == 0) & (df_user_course_info['is_pro']  == 1)]
df_pros_2

,user_id,nickname,is_tutor,is_pro,origin_country_id,living_country_id,origin_city_id,origin_city_name,living_city_id,living_city_name,timezone,trial_length,has_trial,trial_price,min_price,trial_session_count,has_beginner_course
2,114708,Amy (Féng lǎoshī）,0,1,CN,CN,CN00201,Jilin,CN00201,Jilin,Asia/Shanghai,2,0,500,1990,6,1
4,148092,Jhon Jairo,0,1,CO,CO,CO00101,Armenia,CO00006,Medellin,America/Bogota,2,0,1000,2500,182,0
10,346460,Rose,0,1,US,CZ,US01176,Spokane,CZ00000,Other,Europe/Berlin,2,0,1000,1700,175,0
13,399468,Tati Voronkoff,0,1,BR,BR,BR00081,Maceio,BR00081,Maceio,America/Recife,2,0,800,1500,109,0
14,426279,College&Career-Troy,0,1,US,US,US01116,San Jose,US01116,San Jose,America/Los_Angeles,2,0,4100,7500,121,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10448,32102337,Junko.S,0,1,JP,JP,JP00000,Other,JP00000,Other,Asia/Tokyo,2,0,500,1200,0,1
10452,32106641,Elisa,0,1,DE,DE,ES00000,Other,ES00000,Other,Europe/Berlin,2,0,1000,5000,0,0
10457,32113766,Haruka.O,0,1,JP,CZ,JP00151,Nishinomiya,CZ00001,Prague,Europe/Prague,3,0,500,1000,0,0
10459,32117980,Brice,0,1,FR,RO,FR00016,Toulouse,RO00001,Bucharest,Europe/Bucharest,4,0,1000,2600,0,1


In [39]:
df_pros_combined = pd.concat([df_pros_1, df_pros_2], ignore_index=True)
df_pros_combined = df_pros_combined[['user_id']]
df_pros_combined

,user_id
0,113638
1,222305
2,259649
3,291538
4,316582
...,...
4826,32102337
4827,32106641
4828,32113766
4829,32117980


In [40]:
df_merged = pd.merge(df_pros_combined, df_pro_course_english, left_on="user_id", right_on="teacher_id", how="inner")
df_merged

,user_id,id,teacher_id,language,title,session_price,student_count,session_count,has_package
0,222305,40320,222305,english,English,1499,188,730,1
1,222305,40643,222305,english,English,1499,145,492,1
2,436016,375,436016,english,English courses新概念英语（1）,1400,15,106,0
3,463477,4042,463477,english,One-on-One tutoring: English,3200,95,1504,1
4,463477,112222,463477,english,Conversational English,3000,139,782,1
...,...,...,...,...,...,...,...,...,...
7668,32040824,289545,32040824,english,Fun Topics,3000,0,0,1
7669,32040824,289629,32040824,english,Fun Topics 2,3000,0,0,1
7670,32164754,290545,32164754,english,Stock Market English for Traders & Investors (...,4000,0,0,1
7671,32164754,290597,32164754,english,English for Business & Communication (Basic),2000,0,0,1


In [41]:
average_price = {}
median_price = {}

for language in retrieved_languages:
    df_pro_course_language = df_pro_course[df_pro_course['language'] == language]
    df_merged = pd.merge(df_pros_combined, df_pro_course_language, left_on="user_id", right_on="teacher_id", how="inner")
    median = df_merged['session_price'].median()
    mean = df_merged['session_price'].mean()
    average_price[language] = round(mean)/100
    median_price[language] = round(median)/100

df_average_price = pd.DataFrame.from_dict(average_price, orient="index", columns=["Average Price"])
df_average_price
fig = px.bar(df_average_price, y='Average Price')
fig.show()


In [42]:
df_median_price = pd.DataFrame.from_dict(median_price, orient="index", columns=["Median Price"])
df_median_price
fig = px.bar(df_median_price, y='Median Price')
fig.show()

What are the most common languages in the also speaks table?

In [43]:
df_also_speaks

,user_id,language
0,5467830,0
1,5467830,1
2,5467830,2
3,11401921,3
4,11401921,4
...,...,...
29180,23372570,4
29181,23372570,0
29182,8296973,180
29183,8296973,12


In [44]:
df_also_speaks_reference = df_also_speaks_reference.reset_index()
df_also_speaks_reference = df_also_speaks_reference.rename(columns={0:'language'})

In [45]:
df_also_speaks_reference

,index,language
0,0,spanish
1,1,arabic
2,2,arabic(maghrebi)
3,3,filipino(tagalog)
4,4,japanese
...,...,...
205,205,friulian
206,206,luxembourgish
207,207,malagasy
208,208,greenlandic


In [46]:
df_merge_also_speaks = pd.merge(df_also_speaks, df_also_speaks_reference, left_on='language', right_on='index', how="left")
df_merge_also_speaks = df_merge_also_speaks[['user_id', 'language_y']]
df_merge_also_speaks_chart = df_merge_also_speaks.groupby('language_y').count().sort_values('user_id', ascending=False).head(10).reset_index()

In [47]:
df_merge_also_speaks_chart

,language_y,user_id
0,english,6086
1,other,3992
2,spanish,3284
3,french,2586
4,german,1427
5,italian,1366
6,chinese,1272
7,portuguese,1196
8,japanese,1140
9,russian,927


In [48]:
fig = px.bar(df_merge_also_speaks_chart, x='user_id', y='language_y', title='Most Common Languages (non-teaching) Spoken by Teachers', labels={'user_id': 'Number of Teachers', 'language_y': 'Language'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

This chart is interesting, there are a total of 10470 teachers in the dataset.\
An English teacher cannot also put 'English' under the 'also speaks' category.
The difference between the total number of teachers and the number of teachers that list that they speak English is 4384.\
This is close to the number of teachers English teachers on the platform which is 4180.\
This means that with the exception of around 200 teachers in the dataset, every teacher can speak a certain level of English.

What are the typical trial durations and trial lengths?

It is important to note that the teacher only has 1 trial price even if they are teachers of different languages\
There are 11 teachers whose trial price is 0.\
In general the minimum price people set as their trial is $5

In [49]:
len(df_user_course_info[df_user_course_info['trial_price'] < 500])

11

Trial Length distribution

In [50]:
df_trial_length = df_user_course_info.groupby('trial_length')[['user_id']].count()
df_trial_length = df_trial_length.reset_index()
df_trial_length

,trial_length,user_id
0,2,9631
1,3,545
2,4,294


In [51]:
df_trial_length.loc[df_trial_length['trial_length'] == 2, 'trial_length'] = 30
df_trial_length.loc[df_trial_length['trial_length'] == 3, 'trial_length'] = 45
df_trial_length.loc[df_trial_length['trial_length'] == 4, 'trial_length'] = 60
df_trial_length

,trial_length,user_id
0,30,9631
1,45,545
2,60,294


In [52]:
fig = px.bar(df_trial_length, y='user_id', x='trial_length', title='Trial Lengths', labels={'user_id': 'Number of Teachers', 'trial_length': 'Trial Length (Minutes)'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.update_xaxes(tickvals=[30, 45, 60])
fig.show()

Majority of the trial durations are 30 minutes in duration.\
For the 45 mins and the 60 mins, to compare the trial price, take the fraction of the price and keep the price of the 30 mins

In [53]:
df_trial_price = df_user_course_info[['trial_length', 'trial_price']]
df_trial_price.loc[df_trial_price['trial_length'] == 2, 'trial_length'] = 30
df_trial_price.loc[df_trial_price['trial_length'] == 3, 'trial_length'] = 45
df_trial_price.loc[df_trial_price['trial_length'] == 4, 'trial_length'] = 60
df_trial_price[df_trial_price['trial_length'] == 60]


,trial_length,trial_price
83,60,3000
89,60,750
97,60,4800
160,60,750
204,60,2000
...,...,...
10392,60,600
10459,60,1000
10461,60,2500
10462,60,500


In [54]:
def change_price(row):
    if row['trial_length'] == 60:
        return round((row['trial_price'] / 2))
    elif row['trial_length'] == 45:
        return round(((row['trial_price'] / 3) * 2))
    else:
        return round(row['trial_price'])
    
df_trial_price['trial_price_new'] = df_trial_price.apply(change_price, axis=1)


In [55]:
df_trial_price[['trial_price_new']].mean()

trial_price_new    970.523496
dtype: float64

In [56]:
df_trial_price[['trial_price_new']].median()

trial_price_new    800.0
dtype: float64

When standardized to 30 mins of trial prices. They are $9.70 for the mean and $8 for the median

Trial Price by Language

In [57]:
df_user_course_info

,user_id,nickname,is_tutor,is_pro,origin_country_id,living_country_id,origin_city_id,origin_city_name,living_city_id,living_city_name,timezone,trial_length,has_trial,trial_price,min_price,trial_session_count,has_beginner_course
0,55502,Micky Mick,1,0,CN,CN,CN00226,Fushun,CN00006,Beijing,Asia/Shanghai,2,0,1000,2000,97,1
1,113638,Xin,1,1,CN,CA,,,CA00006,Montreal,America/Toronto,2,0,999,2500,227,0
2,114708,Amy (Féng lǎoshī）,0,1,CN,CN,CN00201,Jilin,CN00201,Jilin,Asia/Shanghai,2,0,500,1990,6,1
3,132815,Julie 줄리,1,0,KR,KR,KR00001,Seoul,KR00001,Seoul,Asia/Seoul,2,0,750,1500,0,1
4,148092,Jhon Jairo,0,1,CO,CO,CO00101,Armenia,CO00006,Medellin,America/Bogota,2,0,1000,2500,182,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10465,32167939,Andrele,1,0,CA,CA,CA00000,Other,CA00000,Other,America/Toronto,2,0,1000,2000,3,1
10466,32168520,Daphne👩🏫🌸💗,1,0,PH,PH,PH00141,Baguio,PH00000,Other,Asia/Manila,3,0,500,700,0,0
10467,32169200,Davide Castellani,1,0,IT,IT,IT00000,Other,IT00000,Other,Europe/Rome,4,0,1400,1400,0,1
10468,32169731,FelixRansford,1,0,GB,ES,GB00281,Exeter,ES00131,Jerez de la Frontera,Europe/Madrid,2,0,600,1400,0,0


In [58]:
df_pro_course

,id,teacher_id,language,title,session_price,student_count,session_count,has_package
0,375,436016,english,English courses新概念英语（1）,1400,15,106,0
1,555,436016,chinese,Young Learners' Chinese,1600,5,37,1
2,946,502096,serbian,Serbian for all levels! 100% target language l...,2300,80,1113,1
3,3190,436016,chinese,One-on-One Customized Chinese lesson,1500,73,1391,1
4,4042,463477,english,One-on-One tutoring: English,3200,95,1504,1
...,...,...,...,...,...,...,...,...
39030,290784,31481128,french,🗣️ Natural French Conversation: Speak easily !,700,0,0,1
39031,290785,8513543,french,Leçon d'essai,800,0,0,0
39032,290788,29325526,korean,For Long-term Students,1300,0,0,1
39033,290792,6417813,spanish,DELE exam preparation / Preparación para el ex...,2700,0,0,1


In [59]:
df_user_course_info

,user_id,nickname,is_tutor,is_pro,origin_country_id,living_country_id,origin_city_id,origin_city_name,living_city_id,living_city_name,timezone,trial_length,has_trial,trial_price,min_price,trial_session_count,has_beginner_course
0,55502,Micky Mick,1,0,CN,CN,CN00226,Fushun,CN00006,Beijing,Asia/Shanghai,2,0,1000,2000,97,1
1,113638,Xin,1,1,CN,CA,,,CA00006,Montreal,America/Toronto,2,0,999,2500,227,0
2,114708,Amy (Féng lǎoshī）,0,1,CN,CN,CN00201,Jilin,CN00201,Jilin,Asia/Shanghai,2,0,500,1990,6,1
3,132815,Julie 줄리,1,0,KR,KR,KR00001,Seoul,KR00001,Seoul,Asia/Seoul,2,0,750,1500,0,1
4,148092,Jhon Jairo,0,1,CO,CO,CO00101,Armenia,CO00006,Medellin,America/Bogota,2,0,1000,2500,182,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10465,32167939,Andrele,1,0,CA,CA,CA00000,Other,CA00000,Other,America/Toronto,2,0,1000,2000,3,1
10466,32168520,Daphne👩🏫🌸💗,1,0,PH,PH,PH00141,Baguio,PH00000,Other,Asia/Manila,3,0,500,700,0,0
10467,32169200,Davide Castellani,1,0,IT,IT,IT00000,Other,IT00000,Other,Europe/Rome,4,0,1400,1400,0,1
10468,32169731,FelixRansford,1,0,GB,ES,GB00281,Exeter,ES00131,Jerez de la Frontera,Europe/Madrid,2,0,600,1400,0,0


In [60]:
df_tutors = df_user_course_info[(df_user_course_info['is_tutor'] == 1) & (df_user_course_info['is_pro']  == 0)]
df_tutors[['user_id']]

,user_id
0,55502
3,132815
6,249152
11,355886
12,368484
...,...
10465,32167939
10466,32168520
10467,32169200
10468,32169731


Functions to narrow tutor/pro and languages

In [61]:
def tutor_pro(tutor):
    if tutor == True:
        df_tutors = df_user_course_info[(df_user_course_info['is_tutor'] == 1) & (df_user_course_info['is_pro']  == 0)]
        return df_tutors[['user_id', 'trial_session_count', 'trial_price']]
    else:
        df_pros_1 = df_user_course_info[(df_user_course_info['is_tutor'] == 1) & (df_user_course_info['is_pro']  == 1)]
        df_pros_2 = df_user_course_info[(df_user_course_info['is_tutor'] == 0) & (df_user_course_info['is_pro']  == 1)]
        df_pros_combined = pd.concat([df_pros_1, df_pros_2], ignore_index=True)
        df_pros_combined = df_pros_combined[['user_id', 'trial_session_count', 'trial_price']]
        return df_pros_combined
        
        
tutor_pro(True)

,user_id,trial_session_count,trial_price
0,55502,97,1000
3,132815,0,750
6,249152,70,800
11,355886,142,500
12,368484,91,1000
...,...,...,...
10465,32167939,3,1000
10466,32168520,0,500
10467,32169200,0,1400
10468,32169731,0,600


In [62]:
def teacher_id_language(language):
    df_teachers_id = df_pro_course[df_pro_course['language'] == language].groupby(['teacher_id', 'language']).count()
    df_teachers_id = df_teachers_id.reset_index()
    df_teachers_id = df_teachers_id[['teacher_id']]
    return df_teachers_id

teacher_id_language('chinese')

,teacher_id
0,55502
1,113638
2,114708
3,259649
4,436016
...,...
993,31845535
994,31847889
995,31913484
996,31915782


In [63]:
def pro_course_language(language):
    df_pro_course_target = df_pro_course[df_pro_course['language'] == language]
    return df_pro_course_target

pro_course_language('english')

,id,teacher_id,language,title,session_price,student_count,session_count,has_package
0,375,436016,english,English courses新概念英语（1）,1400,15,106,0
4,4042,463477,english,One-on-One tutoring: English,3200,95,1504,1
6,5779,502096,english,One-on-One tutoring: English,2300,84,876,1
17,10048,596847,english,English language instruction,2600,147,2881,1
21,10779,483024,english,English Pronunciation & Intonation,6500,49,90,1
...,...,...,...,...,...,...,...,...
39014,290713,18491956,english,IELTS Preparation Course,800,0,0,1
39015,290714,18491956,english,"Medical, Health & Fitness English",800,0,0,1
39017,290721,5846271,english,"Beginner English (for FR, ES, PT, ZH speakers)",2200,0,0,1
39019,290736,19100180,english,English Language Topics Lesson,2300,0,0,1


In [64]:
def language_tutor_pro(language, tutor):        
    df_tutor_pro = tutor_pro(tutor)
    df_language = pro_course_language(language)
    df_merged = pd.merge(df_tutor_pro, df_language, left_on='user_id', right_on='teacher_id', how='inner')
    df_merged = df_merged[['user_id', 'trial_session_count', 'trial_price', 'student_count', 'session_count', 'has_package', 'session_price']]
    return df_merged

language_tutor_pro('english', True)

,user_id,trial_session_count,trial_price,student_count,session_count,has_package,session_price
0,484682,0,500,13,238,1,1600
1,495195,440,1000,41,250,1,1400
2,495195,440,1000,20,108,1,1400
3,495195,440,1000,17,122,1,1400
4,544865,0,800,80,934,1,1800
...,...,...,...,...,...,...,...
4616,32168520,0,500,1,1,1,700
4617,32168520,0,500,0,0,1,700
4618,32168520,0,500,0,0,1,800
4619,32169200,0,1400,0,0,0,1400


Trial Prices seem to follow the session prices when split by language.\
German has the highest prices again.

In [65]:
average_price = {}
median_price = {}

for language in retrieved_languages:
    df_teacher_id = teacher_id_language(language)
    df_pro_course_language = df_pro_course[df_pro_course['language'] == language]
    df_merged = pd.merge(df_teacher_id, df_user_course_info, left_on="teacher_id", right_on="user_id", how="inner")
    median = df_merged['trial_price'].median()
    mean = df_merged['trial_price'].mean()
    average_price[language] = round(mean)/100
    median_price[language] = round(median)/100

df_average_price = pd.DataFrame.from_dict(average_price, orient="index", columns=["Average Trial Price"])
df_average_price
fig = px.bar(df_average_price, y='Average Trial Price')
fig.show()


In [66]:
df_median_price = pd.DataFrame.from_dict(median_price, orient="index", columns=["Median Trial Price"])
df_median_price
fig = px.bar(df_median_price, y='Median Trial Price')
fig.show()

In [67]:
df_pro_course

,id,teacher_id,language,title,session_price,student_count,session_count,has_package
0,375,436016,english,English courses新概念英语（1）,1400,15,106,0
1,555,436016,chinese,Young Learners' Chinese,1600,5,37,1
2,946,502096,serbian,Serbian for all levels! 100% target language l...,2300,80,1113,1
3,3190,436016,chinese,One-on-One Customized Chinese lesson,1500,73,1391,1
4,4042,463477,english,One-on-One tutoring: English,3200,95,1504,1
...,...,...,...,...,...,...,...,...
39030,290784,31481128,french,🗣️ Natural French Conversation: Speak easily !,700,0,0,1
39031,290785,8513543,french,Leçon d'essai,800,0,0,0
39032,290788,29325526,korean,For Long-term Students,1300,0,0,1
39033,290792,6417813,spanish,DELE exam preparation / Preparación para el ex...,2700,0,0,1


To consider, chroropleth by timezone, since the city has many unknowns\
Run campaigns to encourage more teachers in certain countries to join the platform.

In [68]:
df_user_course_info.groupby('timezone')[['user_id']].count().sort_values('user_id', ascending=False).head(40)

,user_id
timezone,
Asia/Tokyo,926
Asia/Shanghai,763
Europe/Paris,669
Europe/Berlin,579
Europe/London,532
America/Bogota,519
Europe/Rome,514
Europe/Madrid,427
America/Mexico_City,397


Histogram for the session count for the pro_courses\
After filtering for the courses that are less than 500. The bulk of the courses completed lie within the 0-9 range.\
This means that italki as a platform, could help teachers to scope their courses better.

In [69]:
df_pro_course_filter = df_pro_course[df_pro_course['session_count'] < 500]
fig = px.histogram(df_pro_course_filter, x='session_count', nbins=50, title='Session Count')
# fig = px.box(df_pro_course_filter, y='session_count', title='Session Count')
fig.show()

What about finished sessions at a teacher level?\
At a teacher level, there are 600+ teachers that have only completed less than 10 sessions.\
italki as a platform could help the teachers to onboard better so that the proportion of teachers that complete sessions are lower.

In [70]:
df_teacher_stats_filtered = df_teacher_stats[df_teacher_stats['finished_session'] < 500]
fig = px.histogram(df_teacher_stats_filtered, x='finished_session', nbins=50, title='Session Count')
fig.show()

Find correlation between the various features (See prediction_model.ipynb)

In [71]:
df_user_course_info

,user_id,nickname,is_tutor,is_pro,origin_country_id,living_country_id,origin_city_id,origin_city_name,living_city_id,living_city_name,timezone,trial_length,has_trial,trial_price,min_price,trial_session_count,has_beginner_course
0,55502,Micky Mick,1,0,CN,CN,CN00226,Fushun,CN00006,Beijing,Asia/Shanghai,2,0,1000,2000,97,1
1,113638,Xin,1,1,CN,CA,,,CA00006,Montreal,America/Toronto,2,0,999,2500,227,0
2,114708,Amy (Féng lǎoshī）,0,1,CN,CN,CN00201,Jilin,CN00201,Jilin,Asia/Shanghai,2,0,500,1990,6,1
3,132815,Julie 줄리,1,0,KR,KR,KR00001,Seoul,KR00001,Seoul,Asia/Seoul,2,0,750,1500,0,1
4,148092,Jhon Jairo,0,1,CO,CO,CO00101,Armenia,CO00006,Medellin,America/Bogota,2,0,1000,2500,182,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10465,32167939,Andrele,1,0,CA,CA,CA00000,Other,CA00000,Other,America/Toronto,2,0,1000,2000,3,1
10466,32168520,Daphne👩🏫🌸💗,1,0,PH,PH,PH00141,Baguio,PH00000,Other,Asia/Manila,3,0,500,700,0,0
10467,32169200,Davide Castellani,1,0,IT,IT,IT00000,Other,IT00000,Other,Europe/Rome,4,0,1400,1400,0,1
10468,32169731,FelixRansford,1,0,GB,ES,GB00281,Exeter,ES00131,Jerez de la Frontera,Europe/Madrid,2,0,600,1400,0,0


In [72]:
df_pro_course

,id,teacher_id,language,title,session_price,student_count,session_count,has_package
0,375,436016,english,English courses新概念英语（1）,1400,15,106,0
1,555,436016,chinese,Young Learners' Chinese,1600,5,37,1
2,946,502096,serbian,Serbian for all levels! 100% target language l...,2300,80,1113,1
3,3190,436016,chinese,One-on-One Customized Chinese lesson,1500,73,1391,1
4,4042,463477,english,One-on-One tutoring: English,3200,95,1504,1
...,...,...,...,...,...,...,...,...
39030,290784,31481128,french,🗣️ Natural French Conversation: Speak easily !,700,0,0,1
39031,290785,8513543,french,Leçon d'essai,800,0,0,0
39032,290788,29325526,korean,For Long-term Students,1300,0,0,1
39033,290792,6417813,spanish,DELE exam preparation / Preparación para el ex...,2700,0,0,1


In [73]:
df_merge = pd.merge(df_user_course_info, df_pro_course, left_on='user_id', right_on='teacher_id', how='left')
df_merge_user = df_merge.groupby('user_id')[['student_count', 'session_count', 'session_price']].sum()
df_merge_user = df_merge_user.reset_index()
df_merge_user_zero = df_merge_user[(df_merge_user['student_count'] > 0) & (df_merge_user['session_count'] > 0)]
df_merge_user_zero['session to student'] = df_merge_user_zero['session_count'] / df_merge_user_zero['student_count']
df_merge_user_zero


,user_id,student_count,session_count,session_price,session to student
0,55502,205,2562,6300,12.497561
1,113638,456,7726,12000,16.942982
2,114708,10,44,8360,4.400000
3,132815,235,1980,9300,8.425532
4,148092,273,3897,10800,14.274725
...,...,...,...,...,...
10449,32102519,1,2,1000,2.000000
10451,32106062,3,3,6500,1.000000
10456,32113662,2,2,1200,1.000000
10457,32113766,2,2,4900,1.000000


In [74]:
fig = px.histogram(df_merge_user_zero, x='session to student', nbins=50, title='Session to student')
fig.show()

In [75]:
fig = px.scatter(df_merge_user_zero, x='session_count', y='session_price', title="session to student and session price")
fig.show()

In [76]:
df_merge_user_zero.sort_values('session to student', ascending=False).head(30)

,user_id,student_count,session_count,session_price,session to student
6626,10328459,6,709,1400,118.166667
299,1438812,9,882,6500,98.000000
7576,12575016,5,474,1600,94.800000
5478,8724147,5,469,3600,93.800000
5358,8627195,4,359,1600,89.750000
6363,9783234,93,7693,2500,82.720430
8673,23907744,1,79,7000,79.000000
2377,5643897,60,4210,15800,70.166667
5270,8564380,7,437,6400,62.428571
6933,10899860,76,4724,15600,62.157895


In [78]:
df_user_course_info[df_user_course_info['timezone'] == 'Antarctica/Rothera']

,user_id,nickname,is_tutor,is_pro,origin_country_id,living_country_id,origin_city_id,origin_city_name,living_city_id,living_city_name,timezone,trial_length,has_trial,trial_price,min_price,trial_session_count,has_beginner_course
7543,12447320,Pepe,0,1,MX,MX,MX00000,Other,MX00000,Other,Antarctica/Rothera,2,0,1000,2000,42,1


In [87]:
fig = px.histogram(df_price_list['package_length'], nbins=10, labels={'value': 'Sessions per Package'})
fig.update_layout(legend_title_text = 'Package Length')
fig.data[0].name = "Package Length"
fig.show()